# Pre-train a transaction model, then fine-tune it to flag fraud

A bank has years of unlabeled transaction histories and only a few labeled account
takeovers. We pre-train a model on the histories by having it recover masked field
values, then fine-tune it to predict whether a user will have a fraudulent transaction in
the next seven days. The model follows the design of PRAGMA, a published foundation model
for banking transactions; [model input shapes](../../docs/07-model-input-shapes.md) covers
it in more detail.

Along the way, this notebook shows how to:

- turn every profile value and every transaction field into a token in one shared ID
  space, with a parallel field-name ID saying which field each token came from;
- share one encoder between fields (a single currency vocabulary for the profile's home
  currency and each transaction's currency) while the other fields keep their own;
- pack each transaction's tokens back to back, with offsets;
- declare a masked-value target for pre-training;
- fine-tune by reusing the pre-trained input encoders exactly, fitting only the new label,
  freezing the encoder and training a new head;
- save the artifact, load it back, and score in inference mode.

The encoder, its loss and the fraud head live in
[`transaction_model.py`](transaction_model.py) next to this notebook. Both contracts and
the model's input schema are written next to it too, so you can read them without
running anything.

The data comes from the banking generator in `kumo.pytorch.datasets`: about 400 users and
40,000 transactions, with some account takeovers mixed in.
[Model input shapes](../../docs/07-model-input-shapes.md) compares this batch shape with
three other published sequence models.

In [ ]:
import tempfile
from dataclasses import replace
from pathlib import Path

import numpy as np
import torch
from torch.nn import functional as F
from transaction_model import (
    FraudModel,
    TransactionEncoder,
    is_fraud,
    masked_value_loss,
    rounded,
    write_json,
)

from kumo.pytorch import api, datasets

torch.manual_seed(0)
workdir = Path(tempfile.mkdtemp())
data = datasets.banking(workdir / "banking", seed=0)
print(data.metadata.sizes)
print("fraud positive rate per split:", data.metadata.positive_rate)

## The schema

Two of the generator's five tables are enough here. `is_fraud` is stored on each
transaction; the contract reads it only for the fine-tuning label, from rows after the
cutoff.

In [ ]:
INT64, FLOAT64, STRING = api.DType("int64"), api.DType("float64"), api.DType("string")
DAY = 86_400

schema = api.DataSchema(
    tables=(
        api.TableSpec(
            "users",
            "user_id",
            {
                "user_id": INT64,
                "country": STRING,
                "home_currency": STRING,
                "risk_segment": STRING,
                "birth_year": INT64,
            },
            time_column=api.NON_TEMPORAL,
        ),
        api.TableSpec(
            "transactions",
            "transaction_id",
            {
                "transaction_id": INT64,
                "user_id": INT64,
                "merchant_id": INT64,
                "device_id": INT64,
                "occurred_at": None,
                "amount": FLOAT64,
                "currency": STRING,
                "channel": STRING,
                "is_fraud": INT64,
            },
            time_column="occurred_at",
        ),
    ),
    relations=(
        api.RelationSpec("user_transactions", "users", "user_id", "transactions", "user_id"),
    ),
)
dataset = api.Dataset(
    {name: api.Parquet(str(data.table(name))) for name in ("users", "transactions")}
)
anchors = api.AnchorTable(
    source=api.Parquet(str(data.anchors)),
    entity_id="user_id",
    as_of_time="cutoff",
    sample_id="sample_id",
    split="split",
)

## One sample: a profile and a transaction history

The profile includes one point-in-time aggregate: how many distinct devices the user
transacted from in the last seven days. Each transaction carries its hour of day (a
calendar part) and its age at the cutoff. The fine-tuning label, the number of fraudulent
transactions in the next seven days, is declared here too, so both stages share one sample.

In [ ]:
def transactions(start: int | None, stop: int) -> api.Rows:
    return api.Rows(("user_transactions",), api.TimeWindow("occurred_at", start, stop))


sample = api.SampleSpec(
    entity=api.EntitySpec("users", entity_type="user"),
    entity_values={
        "country": api.Field(column="country"),
        "home_currency": api.Field(column="home_currency"),
        "risk_segment": api.Field(column="risk_segment"),
        "birth_year": api.Field(column="birth_year"),
        "devices_7d": api.Aggregate(transactions(-7 * DAY, 0), "count_distinct", "device_id"),
    },
    events={
        "transactions": api.EventSpec(
            transactions(None, 0),
            fields={
                "amount": api.Field(column="amount"),
                "currency": api.Field(column="currency"),
                "channel": api.Field(column="channel"),
                "merchant": api.Field(column="merchant_id"),
                "time": api.Field(column="occurred_at"),
                "hour": api.Derived(
                    operation=api.CalendarPart(part="hour"), inputs=("time",), dtype=INT64
                ),
                "age_days": api.EventTimeDelta(unit="days", dtype=FLOAT64),
            },
        )
    },
    targets={"fraud_next_week": api.Aggregate(transactions(0, 7 * DAY), "sum", "is_fraud")},
)

## The pre-training batch

- `encodings={"currency": ...}` declares one named vocabulary; both currency fields use it
  through `UseEncoding`, so "EUR" has one ID wherever it appears. Every other field gets its
  own private encoder.
- `id_spaces={"tokens": ...}` numbers every encoder's output in one collision-free space,
  with reserved IDs for padding, unknown, missing and mask.
- `field_names` adds a parallel field-name ID per token.
- `Nested(fields=Packed(), events=Padded())` packs each transaction's tokens and pads the
  transactions per user.
- `EventsOf("history")` delivers each kept transaction's age, aligned with the history.
- `MaskedValuesTarget` corrupts a share of history tokens and supervises their originals.

In [ ]:
ROLES = ("pad", "unknown", "missing", "mask", "entity", "event")


def tokens(*fields: api.Encode) -> api.Fields:
    return api.Fields(fields, row=api.Tokens(), field_names="fields", field_space="tokens")


def token(field: str, encoding: api.ValueEncoding) -> api.Encode:
    return api.Encode(field, encoding, id_space="tokens")


pretraining_batch = api.BatchSpec(
    encodings={"currency": api.Vocabulary()},
    id_spaces={"tokens": api.IdSpace(specials={role: f"[{role.upper()}]" for role in ROLES})},
    field_names={"fields": api.FieldNames()},
    inputs={
        "profile": api.Input(
            api.EntitySource(),
            tokens(
                token("country", api.Vocabulary()),
                token("home_currency", api.UseEncoding("currency")),
                token("risk_segment", api.Vocabulary()),
                token("birth_year", api.Buckets(bins=8)),
                token("devices_7d", api.Vocabulary()),
            ),
            api.Padded(),
        ),
        "history": api.Input(
            api.EventSource(("transactions",), max_events=32, keep="last"),
            tokens(
                token("amount", api.Buckets(bins=16)),
                token("currency", api.UseEncoding("currency")),
                token("channel", api.Vocabulary()),
                token("merchant", api.Vocabulary()),
                token("hour", api.Vocabulary()),
            ),
            api.Nested(fields=api.Packed(), events=api.Padded()),
        ),
        "history_timing": api.Input(
            api.EventsOf("history"), api.Fields((api.Encode("age_days", api.Raw()),)), api.Padded()
        ),
    },
    targets={
        "masked": api.MaskedValuesTarget(
            input="history",
            token_probability=0.15,
            event_probability=0.1,
            key_probability=0.0,
            unknown_probability=0.1,
        )
    },
    batching=api.Batching(max_samples=128),
)
pretraining = api.DataLoaderSpec(schema, sample, pretraining_batch)

The DiskGraph engine supports everything this contract uses, so we run it on DiskGraph
over the Parquet files. If a declaration weren't supported, `check` would name it before
any data is read.

In [ ]:
engine = api.DiskGraphEngine(index=workdir / "index")
issues = api.check(contract=pretraining, dataset=dataset, engine=engine, anchors=anchors)
print(issues or "DiskGraph can execute this contract")

## A small encoder

`TransactionEncoder` in [`transaction_model.py`](transaction_model.py) is ordinary
PyTorch. Each transaction's tokens (value plus field name) are averaged into one vector,
and a Transformer reads the profile tokens and the transaction vectors, each shifted by
its age. The masked head (`masked_value_loss`) scores every token in the shared ID space,
with the embedding tied. To summarize a user, the model concatenates the Transformer's
output for each profile field, the average over all positions, and the output for the
most recent transaction.

## Pre-train

Encoders and buckets are fitted on the train split only. The vocabulary size comes from
the fitted shared ID space.

In [ ]:
with api.prepare(pretraining, dataset, engine) as plan:
    with plan.materialize(anchors, fit_split="train") as task:
        encoder = TransactionEncoder(task.manifest["spaces"]["tokens"]["size"])
        optimizer = torch.optim.AdamW(encoder.parameters(), lr=3e-3)
        for epoch in range(3):
            encoder.train()
            losses = []
            for b in task.batches(
                split="train", epoch=epoch, shuffle=True, seed=0, output_format="model"
            ):
                loss = masked_value_loss(encoder, b)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            print(f"pre-train epoch {epoch}: masked-value loss {np.mean(losses):.3f}")
        pretrained = task.to_artifact()
        pretrained.save(workdir / "pretrained")
        api.save_weights(workdir / "encoder.pt", encoder.state_dict(), task.fitted_state)

## Fine-tune for fraud

The fine-tuning contract is the pre-training contract with a different target: the label
replaces the masked values. Passing `fitted=pretrained.fitted_state.inputs` reuses every
input encoder exactly, so token IDs mean what they meant in pre-training, and only the
target is fitted here (a raw count needs nothing). If the inputs differ, `prepare` raises
an error instead of reusing them.

The encoder is frozen and only a new head (`FraudModel`) is trained.

In [ ]:
finetuning = replace(
    pretraining,
    batch=replace(
        pretraining.batch, targets={"fraud": api.LabelTarget("fraud_next_week", api.Raw())}
    ),
)

encoder = TransactionEncoder(pretrained.manifest["spaces"]["tokens"]["size"])
encoder.load_state_dict(api.load_weights(workdir / "encoder.pt", pretrained.fitted_state))
model = FraudModel(encoder)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-3)

with api.prepare(finetuning, dataset, engine) as plan:
    with plan.materialize(
        anchors, fit_split="train", fitted=pretrained.fitted_state.inputs
    ) as task:
        assert task.fitted_state.inputs == pretrained.fitted_state.inputs
        for epoch in range(8):
            model.train()
            model.encoder.eval()
            losses = []
            for b in task.batches(
                split="train", epoch=epoch, shuffle=True, seed=0, output_format="model"
            ):
                loss = F.binary_cross_entropy_with_logits(model(b), is_fraud(b))
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            print(f"fine-tune epoch {epoch}: loss {np.mean(losses):.4f}")
        task.to_artifact().save(workdir / "fraud")
        api.save_weights(workdir / "fraud.pt", model.state_dict(), task.fitted_state)

## Load the artifact and score in inference mode

A scoring job loads the artifact and the weights and materializes the test anchors with
`inference=True`: nothing after the cutoff is read and batches carry no targets. The
generator's labels are read separately, only to evaluate.

In [ ]:
artifact = api.Artifact.load(workdir / "fraud")
scorer = FraudModel(TransactionEncoder(artifact.manifest["spaces"]["tokens"]["size"]))
scorer.load_state_dict(api.load_weights(workdir / "fraud.pt", artifact.fitted_state))
scorer.eval()

scores: dict[int, float] = {}
with api.prepare(artifact.contract, dataset, engine) as plan:
    with plan.materialize(anchors, fitted=artifact.fitted_state, inference=True) as task:
        with torch.no_grad():
            for b in task.batches(split="test", output_format="model"):
                assert not b.targets
                for sample_id, score in zip(
                    b.metadata["sample_ids"], torch.sigmoid(scorer(b)).tolist(), strict=True
                ):
                    scores[int(sample_id)] = score

labels: dict[int, bool] = {}
with api.prepare(finetuning, dataset, engine) as plan:
    with plan.materialize(anchors, fitted=artifact.fitted_state) as task:
        for b in task.batches(split="test", output_format="model"):
            labels.update(
                zip(
                    (int(s) for s in b.metadata["sample_ids"]),
                    is_fraud(b).bool().tolist(),
                    strict=True,
                )
            )

ids = sorted(scores)
area = datasets.auroc(np.array([labels[i] for i in ids]), np.array([scores[i] for i in ids]))
print(f"test anchors {len(ids)}, frauds {sum(labels.values())}, AUROC {area:.3f}")

## The files next to this notebook

The data, the DiskGraph index, the artifacts and the weights stay in the temporary
directory. Three small files are written next to this notebook:

- `contract.json`: the fine-tuning contract, byte for byte the file the fraud artifact
  stores, with the column types read from the Parquet files. The pre-training contract
  differs only in its target.
- `output_schema.json`: what the model receives, per input and field: its encoder,
  vocabulary or bucket boundaries, and the ID space it maps into. Floats are kept to six
  significant digits so the file is the same on every platform.
- `id_spaces.json`: the shared token ID space: its size and its reserved IDs.

In [ ]:
Path("contract.json").write_text(artifact.contract.to_json() + "\n")
write_json("output_schema.json", rounded(artifact.output_schema))
write_json("id_spaces.json", rounded(artifact.manifest["spaces"]))
written = ("contract.json", "output_schema.json", "id_spaces.json")
print({name: len(Path(name).read_bytes()) for name in written})

## Check

The frozen pre-trained encoder plus a small head separates upcoming fraud well above chance.

In [ ]:
assert area > 0.7, area
assert Path("contract.json").read_text() == (workdir / "fraud" / "contract.json").read_text()
print("checks passed")